In [1]:
import json
import re
import os
from pathlib import Path
from functools import reduce

import pandas as pd
from dotenv import load_dotenv
from groq import Groq
import wbgapi as wb

load_dotenv(Path(".env"), override=True)

client_groq = Groq(api_key=os.getenv("GROQ_API_KEY"))

print(os.getenv("GROQ_API_KEY")[:10])  # verificar que carga bien

gsk_NCLmKk


In [2]:
#Cargamos archivo CSV principal con todos los datos de la encuesta FRA en un DataFrame llamado 'df' y luego imprime sus dimensiones para verificar que se ha cargado correctamente.

df = pd.read_csv("../data/processed/master_fra.csv")

print(df.shape)

(270571, 10)


In [3]:
#Creamos DataFram 'questions' con las preguntas únicas del dataset.
#1. Selecciona las columnas category, question code y question label
#2. Elimina filas duplicadas
#3. Ordena por categoría y código de pregunta

questions = (
    df[["category", "question_code", "question_label"]]
    .drop_duplicates()
    .sort_values(["category", "question_code"])
)

In [4]:
#Generamos el question_inventory.xlsx 

questions.to_excel(
    "../data/processed/question_inventory.xlsx",
    index=False
)

print("✅ question_inventory.xlsx generado")

✅ question_inventory.xlsx generado


In [5]:
#Extraemos el bloque de cada código de preguntas usando una expresión regular y contamos cuántas preguntas hay por bloque
questions["block"] = questions["question_code"].str.extract(r"([a-z]+\d+)")

print(f"✅ Bloques únicos: {questions['block'].nunique()}")

✅ Bloques únicos: 80


In [6]:
#Agrupamos las 800 preguntas por bloque y se queda con la primera question label de cada uno como representativa
blocks = (
    questions
    .groupby("block")
    .agg({
        "question_label": "first"
    })
    .reset_index()
)

print(f"✅ Bloques creados: {blocks.shape[0]}")

✅ Bloques creados: 80


In [7]:
#Generamos 'block_inventory'. 
#Muestra qué pregunta representa cada bloque antes de la clasificación con IA

blocks.to_excel(
    "../data/processed/block_inventory.xlsx",
    index=False
)

print("✅ block_inventory.xlsx generado")

✅ block_inventory.xlsx generado


In [8]:
#Creamos copia de blocks y le añade tres columnas vacías (relevant direction y theme)

block_classification = blocks.copy()

block_classification["relevant"] = ""
block_classification["direction"] = ""
block_classification["theme"] = ""

print(f"✅ block_classification creado: {block_classification.shape}")

✅ block_classification creado: (80, 5)


In [9]:
# Preparar lista numerada de bloques para incluirla en el prompt
# Limitamos la pregunta a 150 caracteres para no sobrecargar el contexto
blocks_list = "\n".join([
    f"{i+1}. block='{row['block']}' | question='{row['question_label'][:150]}'"
    for i, (_, row) in enumerate(blocks.iterrows())
])

# Prompt que enviaremos a la IA con los 80 bloques de una sola vez
# Así usamos 1 request en lugar de 80, evitando agotar la cuota
prompt = f"""
You are helping build an LGBT Acceptance Index from EU survey data.

Classify each of the following {len(blocks)} survey question blocks.

For each block determine:
- relevant: "yes" if it measures LGBTI attitudes, experiences or rights. "no" if purely demographic (age, citizenship, etc.)
- direction: "positive" if high % = more acceptance, "negative" if high % = less acceptance, "neutral" if it doesn't map cleanly
- theme: one short label e.g. "workplace_discrimination", "legal_rights", "social_comfort", "political_hostility", "healthcare", "school_safety"

BLOCKS:
{blocks_list}

Return ONLY a valid JSON array with exactly {len(blocks)} objects in the same order:
[
  {{"block": "a11", "relevant": "no", "direction": "neutral", "theme": "demographics"}},
  ...
]
No markdown, no explanation, just the JSON array.
"""

# Llamada a la API de Groq con Llama 3.3 70B
response = client_groq.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)

# Limpiar la respuesta: eliminar backticks de markdown si los hay
raw = response.choices[0].message.content.strip()
raw = re.sub(r"```json|```", "", raw).strip()

# Parsear el JSON y hacer merge con los bloques originales
classifications = json.loads(raw)
df_class = pd.DataFrame(classifications)
block_classification = blocks.merge(df_class, on="block", how="left")

print(f"✅ Clasificados {len(block_classification)} bloques")

✅ Clasificados 80 bloques


In [10]:
# ============================================================
# CORRECCIONES MANUALES SOBRE LA CLASIFICACIÓN DE LA IA
# ============================================================
# Tras revisar el Excel generado por la IA, se identificaron
# algunos bloques mal clasificados que se corrigen aquí.
# Para añadir más correcciones en el futuro, simplemente
# añadir una nueva entrada al diccionario con el bloque y
# los campos a corregir.

corrections = {
    # Edad de coming out: es un dato personal, no mide aceptación social
    "a13": {"relevant": "no", "direction": "neutral", "theme": "demographics"},
    "a14": {"relevant": "no", "direction": "neutral", "theme": "demographics"},
    
    # Localización de incidentes: no tiene un porcentaje útil para el índice
    "fa1": {"relevant": "no", "direction": "neutral", "theme": "violence_location"},
    "fa2": {"relevant": "no", "direction": "neutral", "theme": "violence_location"},
    "fb1": {"relevant": "no", "direction": "neutral", "theme": "harassment_location"},
    "fb2": {"relevant": "no", "direction": "neutral", "theme": "harassment_location"},
    
    # Apertura personal sobre identidad: mide visibilidad, no aceptación social
    # Se mantiene como relevant=yes pero direction=neutral
    "g1": {"relevant": "yes", "direction": "neutral", "theme": "social_openness"},
    "g2": {"relevant": "yes", "direction": "neutral", "theme": "social_openness"},
    "g3": {"relevant": "yes", "direction": "neutral", "theme": "social_openness"},
    
    # Bienestar emocional: indicador relevante de calidad de vida LGBTI
    "h19": {"relevant": "yes", "direction": "positive", "theme": "wellbeing"},
}

# Aplicar correcciones sobre block_classification
# Para cada bloque corregido, actualizamos solo las columnas especificadas
for block, fields in corrections.items():
    for col, val in fields.items():
        block_classification.loc[block_classification["block"] == block, col] = val

print(f"✅ Correcciones aplicadas: {len(corrections)} bloques ajustados")

# Exportar el Excel final con la clasificación corregida
# Este archivo sirve como referencia para revisión manual futura
block_classification.to_excel("../data/processed/block_classification_ai.xlsx", index=False)
print("✅ Excel actualizado con correcciones")

✅ Correcciones aplicadas: 10 bloques ajustados
✅ Excel actualizado con correcciones


In [11]:
# ============================================================
# CÁLCULO DEL ÍNDICE DE ACEPTACIÓN POR PAÍS Y AÑO
# ============================================================

# 1. Mapa de pesos base (asume dirección positiva)
# Los pesos van de 0 (menos aceptación) a 1 (más aceptación)
answer_weights = {
    # Escala widespread
    "Very widespread":    0.0,
    "Fairly widespread":  0.33,
    "Fairly rare":        0.66,
    "Very rare":          1.0,

    # Escala frecuencia
    "Always":   1.0,
    "Often":    0.66,
    "Rarely":   0.33,
    "Never":    0.0,

    # Escala acuerdo
    "Strongly agree":    1.0,
    "Agree":             0.66,
    "Disagree":          0.33,
    "Strongly disagree": 0.0,

    # Escala cantidad
    "All":    1.0,
    "Most":   0.66,
    "A few":  0.33,

    # Escala apertura
    "Very open": 1.0,

    # Escala cambio
    "Increased a lot":    1.0,
    "Increased a little": 0.66,
    "Stayed the same":    0.5,
    "Decreased a little": 0.33,
    "Decreased a lot":    0.0,

    # Binarias
    "Yes": 1.0,
    "No":  0.0,
}

# Respuestas a excluir del cálculo (no aportan información direccional)
exclude_answers = {
    "Don`t know", "Dont know", "Do not know",
    "Other", "Prefer not to say", "None of the above",
    "Current situation is fine"
}

# 2. Extraer bloque de question_code y convertir percentage a numérico
df["block"] = df["question_code"].str.extract(r"([a-z]+\d+)")
df["percentage"] = pd.to_numeric(df["percentage"], errors="coerce")

# 3. Merge con la clasificación para añadir relevant y direction
df_merged = df.merge(
    block_classification[["block", "relevant", "direction"]],
    on="block",
    how="left"
)

# 4. Filtrar solo bloques relevantes y respuestas con peso conocido
df_filtered = df_merged[
    (df_merged["relevant"] == "yes") &
    (~df_merged["answer"].isin(exclude_answers)) &
    (df_merged["answer"].isin(answer_weights.keys()))
].copy()

# 5. Asignar peso base según respuesta
df_filtered["weight"] = df_filtered["answer"].map(answer_weights)

# 6. Invertir peso si la dirección es negativa
# Ej: "Very widespread" en bloque negativo = mala señal → peso bajo
df_filtered.loc[df_filtered["direction"] == "negative", "weight"] = (
    1 - df_filtered.loc[df_filtered["direction"] == "negative", "weight"]
)

# 7. Calcular score ponderado: weight * (percentage / 100)
df_filtered["score"] = df_filtered["weight"] * (df_filtered["percentage"] / 100)

# ============================================================
# 8. AGREGACIÓN POR BLOQUES Y PAÍS/AÑO
# ============================================================

# Primero: score medio por bloque, país y año
# Así cada bloque pesa igual independientemente de cuántas preguntas tenga
block_scores = (
    df_filtered
    .groupby(["year", "CountryCode", "block"])
    .agg(
        block_score=("score", "mean"),
        n_answers=("score", "count")
    )
    .reset_index()
)

print(f"✅ Bloques calculados: {len(block_scores)}")

# Segundo: índice final como media de todos los bloques por país y año
acceptance_index = (
    block_scores
    .groupby(["year", "CountryCode"])
    .agg(
        acceptance_score=("block_score", "mean"),
        n_blocks=("block", "nunique")
    )
    .reset_index()
)

# Reescalar a 0-100 usando min-max normalización
# Así el país con menos aceptación = 0, el más alto = 100
min_score = acceptance_index["acceptance_score"].min()
max_score = acceptance_index["acceptance_score"].max()

acceptance_index["acceptance_index"] = (
    (acceptance_index["acceptance_score"] - min_score) /
    (max_score - min_score) * 100
).round(2)

acceptance_index = acceptance_index.drop(columns=["acceptance_score"])

print(acceptance_index["acceptance_index"].describe())

print(f"✅ Índice calculado: {len(acceptance_index)} combinaciones país-año")
print(acceptance_index.head(20))

# 9. Exportar a Excel y CSV
acceptance_index.to_excel("../data/processed/acceptance_index.xlsx", index=False)
acceptance_index.to_csv("../data/processed/acceptance_index.csv", index=False)
print("✅ Exportado a Excel y CSV")

✅ Bloques calculados: 1302
count     60.000000
mean      46.052167
std       27.793104
min        0.000000
25%       23.757500
50%       47.070000
75%       72.182500
max      100.000000
Name: acceptance_index, dtype: float64
✅ Índice calculado: 60 combinaciones país-año
    year     CountryCode  n_blocks  acceptance_index
0   2012         Austria        29             76.78
1   2012         Average        29             72.22
2   2012         Belgium        29             84.28
3   2012        Bulgaria        29             51.26
4   2012         Croatia        29             58.62
5   2012          Cyprus        29             49.04
6   2012  Czech Republic        29             74.34
7   2012         Denmark        29             93.46
8   2012         Estonia        28             60.82
9   2012         Finland        29             89.05
10  2012          France        29             76.25
11  2012         Germany        29             81.15
12  2012          Greece        29     

In [12]:
# ============================================================
# DESCARGA AUTOMÁTICA DE INDICADORES DEL BANCO MUNDIAL
# ============================================================
# Se detectan automáticamente los años disponibles en el dataset
# para que funcione sin cambios cuando se añadan años nuevos

years = sorted(df["year"].unique().tolist())
print(f"Años detectados en el dataset: {years}")

# Indicadores seleccionados para el modelo de regresión
indicators = {
    "NY.GDP.PCAP.CD":     "gdp_per_capita",       # PIB per cápita (USD)
    "SI.POV.GINI":        "gini_index",            # Desigualdad (Gini)
    "SE.XPD.TOTL.GD.ZS": "education_spending",    # Gasto en educación (% PIB)
    "SP.URB.TOTL.IN.ZS":  "urbanization_rate",    # Tasa de urbanización
    "SL.UEM.TOTL.ZS":     "unemployment_rate",    # Tasa de desempleo
}

dfs = []

for code, name in indicators.items():
    try:
        raw = wb.data.DataFrame(code, time=years, labels=True)
        raw = raw.reset_index()

        # La API devuelve los años como columnas (YR2012, YR2019...)
        # Las convertimos a filas con melt
        year_cols = [c for c in raw.columns if c.startswith("YR")]
        raw = raw.melt(
            id_vars=["economy", "Country"],
            value_vars=year_cols,
            var_name="year",
            value_name=name
        )

        # Limpiar año: "YR2012" → 2012
        raw["year"] = raw["year"].str.replace("YR", "").astype(int)
        raw = raw.rename(columns={"Country": "CountryName", "economy": "CountryCode_wb"})
        raw = raw[["CountryName", "year", name]]

        dfs.append(raw)
        print(f"✅ {name} descargado ({len(raw)} filas)")

    except Exception as e:
        print(f"⚠️ Error descargando {name}: {e}")

# Merge de todos los indicadores en un único DataFrame
wb_data = reduce(
    lambda left, right: left.merge(right, on=["CountryName", "year"], how="outer"),
    dfs
)

print(f"\n✅ World Bank data: {wb_data.shape}")

# Exportar para uso futuro sin necesidad de volver a descargar
wb_data.to_csv("../data/processed/worldbank_indicators.csv", index=False)
print("✅ Exportado a worldbank_indicators.csv")

Años detectados en el dataset: [2012, 2019]
✅ gdp_per_capita descargado (532 filas)
✅ gini_index descargado (532 filas)
✅ education_spending descargado (532 filas)
✅ urbanization_rate descargado (532 filas)
✅ unemployment_rate descargado (532 filas)

✅ World Bank data: (532, 7)
✅ Exportado a worldbank_indicators.csv


In [13]:
# ============================================================
# MERGE ÍNDICE DE ACEPTACIÓN + INDICADORES BANCO MUNDIAL
# ============================================================

# Mapeo de nombres de países que difieren entre FRA y World Bank
name_mapping = {
    "Czech Republic": "Czechia",
    "Slovakia":       "Slovak Republic",
}

# Crear columna CountryName con los nombres adaptados al formato World Bank
acceptance_index["CountryName"] = acceptance_index["CountryCode"].replace(name_mapping)
acceptance_index["CountryName"] = acceptance_index["CountryName"].where(
    acceptance_index["CountryName"] != acceptance_index["CountryCode"],
    acceptance_index["CountryCode"]
)

# Eliminar filas que no son países reales
acceptance_index = acceptance_index[
    ~acceptance_index["CountryCode"].isin(["EU-28", "Average"])
]

# Cruzar el índice de aceptación con los indicadores del Banco Mundial
df_model = acceptance_index.merge(
    wb_data,
    on=["CountryName", "year"],
    how="left"
)

print(f"✅ Dataset final: {df_model.shape}")
print(f"NaN por columna:\n{df_model.isnull().sum()}")

# Exportar dataset listo para la regresión
df_model.to_csv("../data/processed/dataset_regresion.csv", index=False)
print("✅ Exportado a dataset_regresion.csv")

✅ Dataset final: (58, 10)
NaN por columna:
year                  0
CountryCode           0
n_blocks              0
acceptance_index      0
CountryName           0
gdp_per_capita        0
gini_index            1
education_spending    6
urbanization_rate     0
unemployment_rate     0
dtype: int64
✅ Exportado a dataset_regresion.csv


In [14]:
# ============================================================
# IMPUTACIÓN DE VALORES NULOS
# ============================================================
# gini_index y education_spending tienen algunos NaN.
# Estrategia: rellenar con la media del mismo país entre años.
# Si el país no tiene ningún dato, se usa la media global.

for col in ["gini_index", "education_spending"]:
    # Media por país (entre los años disponibles)
    country_mean = df_model.groupby("CountryCode")[col].transform("mean")
    # Rellenar NaN con la media del país
    df_model[col] = df_model[col].fillna(country_mean)
    # Si sigue habiendo NaN (país sin ningún dato), rellenar con media general
    df_model[col] = df_model[col].fillna(df_model[col].mean())

print(f"NaN restantes:\n{df_model.isnull().sum()}")

# Sobreescribir el CSV con el dataset limpio y listo para la regresión
df_model.to_csv("../data/processed/dataset_regresion.csv", index=False)
print("✅ Dataset final limpio exportado")

NaN restantes:
year                  0
CountryCode           0
n_blocks              0
acceptance_index      0
CountryName           0
gdp_per_capita        0
gini_index            0
education_spending    0
urbanization_rate     0
unemployment_rate     0
dtype: int64
✅ Dataset final limpio exportado


In [15]:
# Ver distribución de respuestas en bloques relevantes
print(df_filtered["answer"].value_counts().head(20))
print()

# Ver score medio por bloque
print(block_scores.groupby("block")["block_score"].mean().sort_values())

answer
No                    11158
Yes                   10605
A few                  5476
All                    5464
Most                   5430
Never                  4316
Rarely                 4315
Often                  4309
Always                 4296
Very widespread        2175
Fairly widespread      2175
Fairly rare            2175
Very rare              2175
Strongly agree         1276
Agree                  1276
Disagree               1276
Strongly disagree      1276
Stayed the same         976
Decreased a little      976
Increased a little      973
Name: count, dtype: int64

block
h14         0.057868
h8          0.062618
c10         0.077831
c8          0.082984
c7          0.092008
ix4         0.104524
g3          0.105068
b1          0.109054
c9          0.110814
tr3         0.111149
e1          0.121009
indg1       0.130085
g2          0.132882
tr8         0.135475
cat4        0.145707
b2          0.145722
tr7         0.146512
c1          0.149727
tr1         0.169821
g